# 시작하기 - 도구 설치

In [ ]:
# STEP 0: 도구 설치 — RDKit(분자), Meeko(도킹용 변환), Vina(도킹), gemmi(구조 정리), py3Dmol(3D 뷰)
!pip -q install rdkit "meeko==0.7.1" vina gemmi py3Dmol
print("설치 완료")

# Ligand Preparation

In [ ]:
# ── 리간드 준비 '자동화' ─────────────────────────────────────────
# 어제는 Ligand 준비를 한 단계씩 했지만, Day 2는 리간드를 여러 개 준비해야 해서
# 함수로 묶습니다. 함수 '안'은 어제 그 순서 그대로예요.
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy
import py3Dmol

def prepare_ligand(smiles, name="ligand", show=True):
    """SMILES 하나 → 도킹용 리간드 PDBQT (반환: 파일 경로)."""
    mol = Chem.MolFromSmiles(smiles)                     # 1) SMILES 파싱
    mol = Chem.AddHs(mol)                                # 2) 수소 추가 (점수함수에 필요)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:   # 3) 3D 좌표 생성
        AllChem.EmbedMolecule(mol, randomSeed=42, useRandomCoords=True)  # 큰 분자 대비 재시도
    AllChem.MMFFOptimizeMolecule(mol)                    # 4) 구조 최적화
    # 5) Meeko: 전하·원자타입·torsion 부여 → PDBQT 문자열
    pdbqt = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(mol)[0])[0]
    path = f"{name}.pdbqt"
    with open(path, "w") as f:
        f.write(pdbqt)
    print(f"리간드 '{name}' 준비 완료  (heavy atom {mol.GetNumHeavyAtoms()}개)")

    if show:   # 결합 정보가 있는 RDKit 구조로 3D 확인 (흰색 = 새로 붙은 수소)
        v = py3Dmol.view(width=420, height=420)
        v.addModel(Chem.MolToMolBlock(Chem.RemoveHs(mol)), "mol")
        v.setStyle({"stick": {}}); v.setBackgroundColor("white"); v.zoomTo(); v.show()
    return path

In [ ]:
# 오늘의 리간드: Imatinib (글리벡) — Abl 키나제를 막는 백혈병 치료제
IMATINIB = "Cc1ccc(cc1Nc1nccc(n1)-c1cccnc1)NC(=O)c1ccc(cc1)CN1CCN(C)CC1"
ligand_pdbqt = prepare_ligand(IMATINIB, "imatinib",)     # 3D 뷰로 확인

# Protein Preparation

In [ ]:
# ── 단백질(receptor) 준비 '자동화' ──────────────────────────────
# 어제 여러 셀로 했던 [다운로드 → 청소 → PDBQT 변환 → 탐색 박스]를 함수 하나로.
# 공결정 리간드를 자동 감지해 박스를 잡고(dimer 안전), 나중 RMSD 채점을 위해 그 리간드 정보도 남깁니다.
import urllib.request, subprocess, os, gemmi, numpy as np, py3Dmol

# 단백질이 아니라 무시할 것들 (물·이온·완충제)
SKIP = {"HOH","WAT","NA","CL","CA","MG","ZN","K","MN","FE","CU","SO4","PO4",
        "GOL","EDO","ACT","DMS","PEG"}

def prepare_receptor(pdb_id, show=True):
    """PDB ID → receptor PDBQT + 탐색 박스.
       반환: dict(receptor, box_center, box_size, 공결정 리간드 정보)."""
    raw, clean = f"{pdb_id}.pdb", f"{pdb_id}_clean.pdb"
    receptor   = f"{pdb_id}_receptor.pdbqt"

    # 1) 실험 구조 다운로드
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", raw)
    st = gemmi.read_structure(raw); st.setup_entities()

    # 2) 공결정 리간드(단일 인스턴스) 감지 → 탐색 박스 (어제 박스 셀 로직 그대로, dimer 안전)
    inst = []
    for ch in st[0]:
        for res in ch:
            if res.name in SKIP: continue
            info = gemmi.find_tabulated_residue(res.name)
            if info and (info.is_amino_acid() or info.is_nucleic_acid()): continue
            inst.append(([[a.pos.x,a.pos.y,a.pos.z] for a in res], ch.name, res.name, res.seqid.num))
    coords, lig_chain, lig_name, lig_seq = max(inst, key=lambda t: len(t[0]))   # 원자 많은 '단일' 리간드
    xyz = np.array(coords)
    box_center = xyz.mean(0).round(2).tolist()
    box_size   = (xyz.max(0) - xyz.min(0) + 8).round(1).tolist()   # 리간드 크기에 맞춘 박스(+8Å 여유)

    # 3) 청소 (어제 gemmi 청소 그대로) → clean.pdb
    st.remove_alternative_conformations(); st.remove_hydrogens()
    st.remove_ligands_and_waters(); st.remove_empty_chains(); st.write_pdb(clean)

    # 4) Meeko: 수소·전하·원자타입 부여 → receptor PDBQT
    subprocess.run(["mk_prepare_receptor.py","--read_pdb",clean,"-o",f"{pdb_id}_receptor","-p"],
                   capture_output=True, text=True)
    if not os.path.exists(receptor):
        raise RuntimeError(f"[{pdb_id}] receptor 생성 실패 — mk_prepare_receptor 확인 필요")
    print(f"✅ [{pdb_id}] receptor 준비 완료 · 공결정 리간드 {lig_name} · 박스 중심 {box_center}, 크기 {box_size}")

    if show:   # 단백질(회색) + 공결정 리간드(주황) + 탐색 박스(주황 반투명)
        v = py3Dmol.view(width=560, height=480)
        v.addModel(open(clean).read(), "pdb"); v.setStyle({"cartoon": {"color": "lightgray"}})
        lb = "".join(l for l in open(raw) if l.startswith("HETATM")
                     and l[17:20].strip()==lig_name and l[21]==lig_chain
                     and l[22:26].strip()==str(lig_seq))
        v.addModel(lb, "pdb"); v.setStyle({"model":1}, {"stick":{"colorscheme":"orangeCarbon"}})
        cx,cy,cz = box_center; sx,sy,sz = box_size
        v.addBox({"center":{"x":cx,"y":cy,"z":cz},
                  "dimensions":{"w":sx,"h":sy,"d":sz}, "color":"orange","opacity":0.3})
        v.zoomTo(); v.setBackgroundColor("white"); v.show()

    return {"pdb": pdb_id, "receptor": receptor, "clean": clean, "raw": raw,
            "box_center": box_center, "box_size": box_size,
            "lig_name": lig_name, "lig_chain": lig_chain, "lig_seq": lig_seq}

In [ ]:
# 복습 표적: 1IEP (Abl 키나제 + 이매티닙). 회색=단백질, 주황=공결정 이매티닙, 주황상자=탐색 박스
target = prepare_receptor("1IEP")
print("반환값 keys:", list(target))

# Imatinib docking: 1IEP

In [ ]:
# 준비물(target = 1IEP receptor+박스, ligand_pdbqt = 이매티닙)로 도킹
from vina import Vina

v = Vina(sf_name="vina", seed=42, verbosity=0)          # seed 고정 → 재현 가능
v.set_receptor(target["receptor"])
v.set_ligand_from_file(ligand_pdbqt)
v.compute_vina_maps(center=target["box_center"],        # 준비 때 잡은 그 박스
                    box_size=target["box_size"])
v.dock(exhaustiveness=8, n_poses=5)
v.write_poses("imatinib_docked.pdbqt", n_poses=5, overwrite=True)

print("mode |  affinity (kcal/mol)")
print("-----+--------------------")
for i, row in enumerate(v.energies(n_poses=5), 1):
    print(f"  {i}  |   {row[0]:8.2f}")
print("\n낮을수록(더 음수) 강한 결합. 1위가 Vina의 최선의 추측.")
# → 1위 약 -12.4 kcal/mol

# Imatinib docking: 2GQG

In [ ]:
# ── 단백질 준비 (개정): -a(빠진원자 잔기 삭제) + 실패 시 원인 출력 ──
import urllib.request, subprocess, os, gemmi, numpy as np, py3Dmol

SKIP = {"HOH","WAT","NA","CL","CA","MG","ZN","K","MN","FE","CU","SO4","PO4",
        "GOL","EDO","ACT","DMS","PEG"}

def prepare_receptor(pdb_id, show=False, allow_bad_res=True):
    """PDB ID → receptor PDBQT + 탐색 박스. 반환: dict(receptor, box_center, box_size, 리간드 정보)."""
    raw, clean = f"{pdb_id}.pdb", f"{pdb_id}_clean.pdb"
    receptor   = f"{pdb_id}_receptor.pdbqt"

    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", raw)
    st = gemmi.read_structure(raw); st.setup_entities()

    # 공결정 리간드(단일 인스턴스) 감지 → 탐색 박스 (dimer 안전)
    inst = []
    for ch in st[0]:
        for res in ch:
            if res.name in SKIP: continue
            info = gemmi.find_tabulated_residue(res.name)
            if info and (info.is_amino_acid() or info.is_nucleic_acid()): continue
            inst.append(([[a.pos.x,a.pos.y,a.pos.z] for a in res], ch.name, res.name, res.seqid.num))
    coords, lig_chain, lig_name, lig_seq = max(inst, key=lambda t: len(t[0]))
    xyz = np.array(coords)
    box_center = xyz.mean(0).round(2).tolist()
    box_size   = (xyz.max(0) - xyz.min(0) + 8).round(1).tolist()

    # 청소 → clean.pdb
    st.remove_alternative_conformations(); st.remove_hydrogens()
    st.remove_ligands_and_waters(); st.remove_empty_chains(); st.write_pdb(clean)

    # Meeko: -a = 원자 빠진 잔기(disorder된 loop 등)는 지우고 진행 → 활성형 구조에서 안전
    cmd = ["mk_prepare_receptor.py", "--read_pdb", clean, "-o", f"{pdb_id}_receptor", "-p"]
    if allow_bad_res:
        cmd.append("-a")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if not os.path.exists(receptor):
        print("⚠️ mk_prepare_receptor STDERR:\n", r.stderr[-2500:])   # ← 진짜 원인 표시
        raise RuntimeError(f"[{pdb_id}] receptor 생성 실패 — 위 STDERR 확인")
    print(f"✅ [{pdb_id}] receptor 준비 완료 · 공결정 리간드 {lig_name} · 박스 중심 {box_center}, 크기 {box_size}")

    if show:
        v = py3Dmol.view(width=560, height=480)
        v.addModel(open(clean).read(), "pdb"); v.setStyle({"cartoon": {"color": "lightgray"}})
        lb = "".join(l for l in open(raw) if l.startswith("HETATM")
                     and l[17:20].strip()==lig_name and l[21]==lig_chain
                     and l[22:26].strip()==str(lig_seq))
        v.addModel(lb, "pdb"); v.setStyle({"model":1}, {"stick":{"colorscheme":"orangeCarbon"}})
        cx,cy,cz = box_center; sx,sy,sz = box_size
        v.addBox({"center":{"x":cx,"y":cy,"z":cz},
                  "dimensions":{"w":sx,"h":sy,"d":sz}, "color":"orange","opacity":0.3})
        v.zoomTo(); v.setBackgroundColor("white"); v.show()

    return {"pdb": pdb_id, "receptor": receptor, "clean": clean, "raw": raw,
            "box_center": box_center, "box_size": box_size,
            "lig_name": lig_name, "lig_chain": lig_chain, "lig_seq": lig_seq}

In [ ]:
from rdkit import Chem
from meeko import PDBQTMolecule, RDKitMolCreate

target2 = prepare_receptor("2GQG")                    # 이미 돌렸으면 생략 가능

v2 = Vina(sf_name="vina", seed=42, verbosity=0)
v2.set_receptor(target2["receptor"])
v2.set_ligand_from_file(ligand_pdbqt)                 # 이매티닙
v2.compute_vina_maps(center=target2["box_center"], box_size=target["box_size"])
v2.dock(exhaustiveness=8, n_poses=5)
v2.write_poses("imatinib_2gqg_docked.pdbqt", n_poses=5, overwrite=True)
print(f"[2GQG · DFG-in] 도킹 1위 = {v2.energies(n_poses=1)[0][0]:.2f} kcal/mol")

pmol2   = PDBQTMolecule.from_file("imatinib_2gqg_docked.pdbqt", skip_typing=True)
docked2 = Chem.RemoveHs(RDKitMolCreate.from_pdbqt_mol(pmol2)[0])
pose2   = Chem.MolToMolBlock(docked2, confId=0)
print("pose2 준비 완료 → 블록 10에서 나란히 비교")

# Imatinib - Comparison


In [ ]:
from rdkit import Chem
from meeko import PDBQTMolecule, RDKitMolCreate
import py3Dmol

def load_pose(f):
    pmol = PDBQTMolecule.from_file(f, skip_typing=True)
    return Chem.MolToMolBlock(Chem.RemoveHs(RDKitMolCreate.from_pdbqt_mol(pmol)[0]), confId=0)

def show_pocket(protein_pdb, pose, title=""):
    v = py3Dmol.view(width=680, height=540)
    v.addModel(open(protein_pdb).read(), "pdb")
    v.addModel(pose, "mol")
    v.setStyle({"cartoon": {"color": "lightgray"}})
    v.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    v.addSurface(py3Dmol.VDW, {"opacity": 0.6, "color": "white"},
                 {"within": {"distance": 8, "sel": {"model": 1}}})
    v.zoomTo({"model": 1}); v.setBackgroundColor("white")
    print(title); v.show()

# 파일에서 pose 불러오기 (블록 6·9가 만든 파일)
pose1 = load_pose("imatinib_docked.pdbqt")           # 1IEP
show_pocket(target["clean"], pose1, "1IEP (DFG-out): 이매티닙")

show_pocket(target2["clean"], pose2, "2GQG (DFG-in): 이매티닙")

# Dasatinib - Comparison

In [ ]:
def dock_into(t, ligand_pdbqt, exhaustiveness=4):          # 기본 4로 낮춤
    v = Vina(sf_name="vina", seed=42, verbosity=0)
    v.set_receptor(t["receptor"])
    v.set_ligand_from_file(ligand_pdbqt)
    v.compute_vina_maps(center=t["box_center"], box_size=COMMON_BOX)
    v.dock(exhaustiveness=exhaustiveness, n_poses=1)        # 점수만 → n_poses=1
    return float(v.energies(n_poses=1)[0][0])

In [ ]:
COMMON_BOX = target["box_size"]     # ← 이 줄 추가 (공통 박스 = 1IEP 기준, 공정 비교)

DASATINIB = "CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO"
dasa = prepare_ligand(DASATINIB, "dasatinib")

def load_score(f):
    for l in open(f):
        if l.startswith("REMARK VINA RESULT"): return float(l.split()[3])

# imatinib: 이미 도킹된 파일에서 점수만 (재도킹 X)
imat_1iep = load_score("imatinib_docked.pdbqt")
imat_2gqg = load_score("imatinib_2gqg_docked.pdbqt")

# dasatinib만 새로 도킹 (2번)
dasa_1iep = dock_into(target,  dasa)
dasa_2gqg = dock_into(target2, dasa)

print("            1IEP(DFG-out)   2GQG(DFG-in)")
print(f"imatinib      {imat_1iep:8.2f}       {imat_2gqg:8.2f}")
print(f"dasatinib     {dasa_1iep:8.2f}       {dasa_2gqg:8.2f}")

# Decoys

In [ ]:
ACTIVES = {
 "imatinib":  "Cc1ccc(cc1Nc1nccc(n1)-c1cccnc1)NC(=O)c1ccc(cc1)CN1CCN(C)CC1",
 "nilotinib": "Cc1ccc(cc1Nc1nccc(n1)-c1cccnc1)C(=O)Nc1cc(cc(c1)C(F)(F)F)n1cc(C)nc1",
 "ponatinib": "Cc1ccc(cc1C#Cc1cnc2cccnn12)C(=O)Nc1ccc(CN2CCN(C)CC2)c(c1)C(F)(F)F",
}
DECOYS = {
 "warfarin":"CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O",
 "diclofenac":"OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl",
 "indomethacin":"COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1",
 "sulfasalazine":"OC(=O)c1cc(ccc1O)N=Nc1ccc(cc1)S(=O)(=O)Nc1ccccn1",
 "chlorpromazine":"CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21",
 "loratadine":"CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",
 "furosemide":"NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl",
}
LABELS = {**{k:"active" for k in ACTIVES}, **{k:"decoy" for k in DECOYS}}

# 강사 precompute: 1IEP 도킹 점수 (exhaustiveness 4) — 학생에게 '주어지는' 표
IEP_SCORES = {
 "nilotinib":-13.77, "imatinib":-12.43, "ponatinib":-11.13, "sulfasalazine":-10.10,
 "warfarin":-9.41, "diclofenac":-8.84, "indomethacin":-7.45,
 "loratadine":-6.82, "furosemide":-6.65, "chlorpromazine":-6.39,
}

def enrichment(scores):
    """점수 낮을수록 좋음. ROC-AUC + top-3 안의 active 수 반환."""
    act=[s for n,s in scores.items() if LABELS[n]=="active"]
    dec=[s for n,s in scores.items() if LABELS[n]=="decoy"]
    auc=sum((a<d)+0.5*(a==d) for a in act for d in dec)/(len(act)*len(dec))
    top3=sum(1 for n,_ in sorted(scores.items(), key=lambda x:x[1])[:3] if LABELS[n]=="active")
    return auc, top3

auc1, top1 = enrichment(IEP_SCORES)
print(f"[1IEP · DFG-out] ROC-AUC={auc1:.2f} · actives top-3={top1}/3  (강사 제공)")

In [ ]:
gqg_scores = {}
for name, smi in {**ACTIVES, **DECOYS}.items():
    lig = prepare_ligand(smi, name)              # show=False
    gqg_scores[name] = dock_into(target2, lig)   # 헬퍼(exhaustiveness=4), 2GQG
    print(f"  {name:14s} {LABELS[name]:6s} {gqg_scores[name]:6.2f}")

auc2, top2 = enrichment(gqg_scores)

print("\n===== Template 비교 =====")
print(f"1IEP (DFG-out): ROC-AUC {auc1:.2f} · top-3 {top1}/3   ← 강사 precompute")
print(f"2GQG (DFG-in) : ROC-AUC {auc2:.2f} · top-3 {top2}/3   ← 내가 계산")
print("→ ROC-AUC 높은 쪽이 actives를 잘 골라내는 '믿을 만한 template'")

# AlphaFold

In [ ]:
import json, urllib.request, numpy as np, difflib, subprocess
from vina import Vina

info = json.load(urllib.request.urlopen("https://alphafold.ebi.ac.uk/api/prediction/P00519"))
urllib.request.urlretrieve(info[0]["pdbUrl"], "AF_raw.pdb"); print("AF:", info[0]["pdbUrl"])

open("AF_kd.pdb","w").write("".join(
    l for l in open("AF_raw.pdb")
    if l.startswith("ATOM") and l[21]=="A" and 242 <= int(l[22:26]) <= 512))

t2o={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M'}
def ca(p, one_chain=False):
    recs={}; order=[]
    for l in open(p):
        if not l.startswith(("ATOM","HETATM")): continue
        k=(l[21], l[22:26].strip())
        if k not in recs: recs[k]=(l[17:20].strip(),{}); order.append(k)
        recs[k][1][l[12:16].strip()]=(float(l[30:38]),float(l[38:46]),float(l[46:54]))
    if one_chain:
        first=order[0][0]; order=[k for k in order if k[0]==first]
    seq=[]; xyz=[]
    for k in order:
        rn,at=recs[k]
        if rn in t2o and 'CA' in at: seq.append(t2o[rn]); xyz.append(at['CA'])
    return "".join(seq), np.array(xyz)

sA,cA = ca("AF_kd.pdb")
s1,c1 = ca(target["clean"], one_chain=True)
sm=difflib.SequenceMatcher(None,sA,s1,autojunk=False); P=[];Q=[]
for a,b,n in sm.get_matching_blocks():
    for k in range(n): P.append(cA[a+k]); Q.append(c1[b+k])
P=np.array(P); Q=np.array(Q)

def kabsch(A,B):
    Am,Bm=A.mean(0),B.mean(0)
    U,S,Vt=np.linalg.svd((A-Am).T@(B-Bm))
    R=Vt.T@np.diag([1,1,np.sign(np.linalg.det(Vt.T@U.T))])@U.T
    return lambda x:(x-Am)@R.T+Bm

xform=kabsch(P,Q)
for _ in range(8):
    d=np.sqrt(((xform(P)-Q)**2).sum(1))
    keep = d<2.0
    if keep.sum()<40: keep = d<=np.percentile(d,60)
    xform=kabsch(P[keep], Q[keep])
core=int(keep.sum()); rmsd=round(float(np.sqrt(((xform(P[keep])-Q[keep])**2).sum(1).mean())),2)
print(f"superpose: core {core}/{len(P)} CA · RMSD {rmsd} Å  →  {'✅ OK' if rmsd<3 else '❌ 여전히 실패'}")

out=[]
for l in open("AF_kd.pdb"):
    if l.startswith("ATOM"):
        p=xform(np.array([[float(l[30:38]),float(l[38:46]),float(l[46:54])]]))[0]
        l=l[:30]+f"{p[0]:8.3f}{p[1]:8.3f}{p[2]:8.3f}"+l[54:]
    out.append(l)
open("AF_aligned.pdb","w").writelines(out)

subprocess.run(["mk_prepare_receptor.py","--read_pdb","AF_aligned.pdb","-o","AF_receptor","-p","-a"],
               capture_output=True, text=True)

af_scores={}
for name,smi in {**ACTIVES,**DECOYS}.items():
    lig=prepare_ligand(smi,name)
    v=Vina(sf_name="vina",seed=42,verbosity=0); v.set_receptor("AF_receptor.pdbqt"); v.set_ligand_from_file(lig)
    v.compute_vina_maps(center=target["box_center"], box_size=target["box_size"]); v.dock(exhaustiveness=4,n_poses=1)
    af_scores[name]=float(v.energies(n_poses=1)[0][0])

print("\n[AF] 순위 (best first)")
for i,(n,s) in enumerate(sorted(af_scores.items(),key=lambda x:x[1]),1):
    print(f" {i:2d}. {n:14s} {LABELS[n]:6s} {s:6.2f}")
act=[s for n,s in af_scores.items() if LABELS[n]=="active"]; dec=[s for n,s in af_scores.items() if LABELS[n]=="decoy"]
ma=round(sum(act)/len(act),2); gap=round(sum(dec)/len(dec)-ma,2)
print(f"→ AF: mean_active {ma} · gap {gap}   (1IEP −12.44 / 2GQG −11.02)")